# Contrato lógico de ingestão BigQuery

## Escopo e limite de evidência

Este catálogo documenta os campos lógicos que uma futura ingestão precisa
receber. Ele **não** consulta BigQuery e não supõe projeto, dataset, tabela,
coluna, partição, watermark ou regra de atualização. Tudo que depende do schema
físico permanece como `não fornecido` até validação da Globo.

As fontes ServiceNow e NPAW/Youbora são domínios documentados via BigQuery. Os
seis campos de `frontend_events_sample` usam
`source_system="amostra frontend; origem BigQuery não comprovada"`: vêm de
exemplos de frontend e ficam separados do contrato confirmado.

In [ ]:
from dump_explorer.contract import (
    FRONTEND_SAMPLE_SOURCE,
    OUTPUTS_DIR,
    UNKNOWN_PHYSICAL_MAPPING,
    build_logical_contract,
    export_contract_csv,
    validate_physical_mapping,
)

contract = build_logical_contract()
physical_columns = [
    "bigquery_table",
    "bigquery_column",
    "bigquery_partition_column",
    "bigquery_write_watermark",
    "update_semantics",
]

## 1. Campos lógicos de domínios documentados via BigQuery

NPAW tem atraso estrutural de aproximadamente 28 horas. Em ServiceNow, `opened_at` é ticket-open time, não o início do incidente.

In [ ]:
confirmed_domains = contract.loc[
    contract["source_system"] != FRONTEND_SAMPLE_SOURCE
]
display(confirmed_domains)

domain_summary = confirmed_domains.groupby(
    ["source_system", "logical_table", "grain", "known_delay"], as_index=False
).size()
display(domain_summary)

## 2. Amostras de eventos de frontend

A seção abaixo não é um mapeamento BigQuery e não autoriza ingestão de identificadores individuais. Ela registra apenas os campos lógicos selecionados dos seis exemplos fornecidos.

In [ ]:
frontend_samples = contract.loc[
    contract["source_system"] == FRONTEND_SAMPLE_SOURCE
]
display(frontend_samples)

## 3. Lacunas do mapeamento físico

A validação não converte nomes do dump MySQL em nomes físicos BigQuery. Ela torna explícitas as lacunas que bloqueiam uma ingestão incremental segura e idempotente.

In [ ]:
try:
    unresolved_physical_fields = contract.loc[
        contract[physical_columns].eq(UNKNOWN_PHYSICAL_MAPPING).any(axis=1)
    ]
    issues = validate_physical_mapping(contract)
    display(unresolved_physical_fields)
    display([issue.__dict__ for issue in issues])
    
    if len(issues) > 0:
        print(f"⚠️  Encontrados {len(issues)} problemas de mapeamento físico que bloqueiam ingestão.")
    else:
        print("✅ Nenhum problema de mapeamento físico encontrado.")
except Exception as e:
    print(f"❌ Erro ao validar mapeamento físico: {e}")

## 4. Exportação explícita

Nenhum CSV é criado ao abrir ou executar este notebook. Para exportar, altere conscientemente a flag abaixo; o destino é restringido a `analysis/outputs`.

In [ ]:
export_csv = False

if export_csv:
    try:
        output_path = OUTPUTS_DIR / "contrato_logico_ingestao.csv"
        exported_path = export_contract_csv(contract, output_path)
        print(f"✅ Contrato exportado para {exported_path}")
    except Exception as e:
        print(f"❌ Erro ao exportar contrato: {e}")
else:
    print("ℹ️  Para exportar o contrato, altere export_csv = True e execute esta célula novamente.")

## Perguntas exatas para enviar à Globo

1. Qual é o **project/dataset/table** físico para cada domínio?
2. Qual é a **partition column** de cada tabela?
3. Qual é o **write watermark** confiável?
4. Qual é a **natural key** de idempotência?
5. Como são tratados **late arrivals**?
6. Como são tratados **corrections/deletes**?
7. Como a **schema evolution** é comunicada e compatibilizada?
8. Quais campos têm **PII** e quais regras de sanitização se aplicam?
9. Quais são as **query quotas** e limites de custo/leitura?